# **Echantillonnage et Estimation**

## Statistiques échantillonnales d'un caractère quantitatif

**Exemple 2.1**

Un constructeur automobile fabrique des moteurs diesel pour un certain modèle de voiture. La population est donc constituée de tous les moteurs produits.
L’ingénieur qualité s’intéresse à la pollution en NOx des moteurs.
Il prélève un échantillon de taille 7 dont la réalisation fournit les résultats suivants : (78, 82, 81, 82 ,79, 80, 78).

Autrement dit, $x_1 = 78$, $x_2 = 82$, $x_3 = 81$, $x_4 = 82$, $x_5 = 79$, $x_6 = 80$, $x_7 = 78$.

Des lettres minuscules sont utilisées car il s’agit de la réalisation de l’échantillon.

La réalisation de la moyenne échantillonnale est :

````{dropdown} Solution (cliquez pour afficher)
```{code-cell} python
# Code exécutable avec Thebelab
import numpy as np
valeurs = [78, 82, 81, 82, 79, 80, 78]
print(np.mean(valeurs))
```
````

In [ ]:
import numpy as np
valeurs=[78,82,81,82,79,80,78]
print(np.mean(valeurs))

La réalisation de la variance échantillonnale est :

In [ ]:
print(np.var(valeurs))

## Statistiques échantillonnales d'un caractère qualitatif

**Exemple 2.2**

Une entreprise automobile s’intéresse à une pièce fondamentale constitutive des moteurs produits.

Elle prélève un échantillon de taille 7 dont la réalisation fournit les résultats suivants : $(1,0,0,1,0,0,0)$ où $1$ signifie que la pièce est défectueuse et $0$ sinon.

Autrement dit, $x_1 = 1$, $x_2 = 0$, $x_3 = 0$, $x_4 = 1$, $x_5 = 0$, $x_6 = 0$, $x_7 = 0$.

La réalisation de la variable fréquence est ainsi :

In [ ]:
valeurs=[1,0,0,1,0,0,0]
n=len(valeurs)
frequence=np.sum(valeurs)/n
print(f"{frequence:.1%}")

**Exemple 2.3**

Supposons que tous les matins on prenne le bus n◦2 de 7h20 et que l’on veuille estimer notre temps d’attente maximum de ce bus.

On suppose que ce temps d’attente (en s) est une variable aléatoire $X$ suivant
une loi uniforme sur $[0; \theta]$.

On choisit $n$ jours au hasard et on relève les temps d’attente $X_1,..., X_n$.

On dispose donc d’un échantillon $(X_1,..., X_n)$.

Proposer, à partir de cet échantillon, plusieurs estimateurs de $\theta$, le temps d’attente maximum.

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Paramètres
n_samples = 100
sample_size = 5
a, b = 0, 360
theta = 360

# Simulation
np.random.seed(42)
samples = np.random.uniform(a, b, size=(n_samples, sample_size))

moyennes = np.mean(samples, axis=1)
max_values = np.max(samples, axis=1)
min_values = np.min(samples, axis=1)

estimator_1 = 2 * moyennes
estimator_2 = max_values
estimator_3 = ((sample_size + 1) / sample_size) * max_values
estimator_4 = max_values + min_values

x_min, x_max = 1, n_samples
y_min = min(np.min(estimator_1), np.min(estimator_2), np.min(estimator_3), np.min(estimator_4)) - 10
y_max = max(np.max(estimator_1), np.max(estimator_2), np.max(estimator_3), np.max(estimator_4)) + 10

data_list = [estimator_1, estimator_2, estimator_3, estimator_4]
titres = [r"$\hat{\theta}_{5,1}$", r"$\hat{\theta}_{5,2}$", r"$\hat{\theta}_{5,3}$", r"$\hat{\theta}_{5,4}$"]
couleurs = ['blue', 'green', 'orange', 'purple']

# Figure
plt.close('all')  # évite les figures fantômes en cas de ré-exécution
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle(r"Estimations des 100 échantillons de taille 5 (Loi Uniforme $[0, \theta]$)", fontsize=13)

scatters = []
for ax, titre, couleur in zip(axes.flat, titres, couleurs):
    ax.set_title(titre)
    ax.set_xlabel("Numéro d'échantillon")
    ax.set_ylabel("Estimation")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    sc = ax.scatter([], [], color=couleur, s=10)
    scatters.append(sc)

plt.tight_layout(rect=[0, 0, 1, 0.94])

# --- Widgets de contrôle ---
bouton = widgets.Button(description="Suivant ▶", button_style='success')
info_label = widgets.Label(value="Cliquez pour afficher le point 1")
display(widgets.HBox([bouton, info_label]))

# --- État de l'animation ---
etat = {"frame": 0, "phase": "manuel"}  # phase : 'manuel', 'auto_lance', 'droite', 'fini'
seuil_manuel = 10

def afficher_point(frame):
    x = np.arange(1, frame + 2)
    for sc, data in zip(scatters, data_list):
        y = data[:frame + 1]
        sc.set_offsets(np.column_stack((x, y)))
    fig.canvas.draw_idle()

def phase_automatique():
    """Enchaîne les points restants automatiquement avec un léger délai."""
    import time
    for frame in range(seuil_manuel, n_samples):
        afficher_point(frame)
        fig.canvas.draw_idle()
        fig.canvas.flush_events()
        time.sleep(0.05)
    etat["phase"] = "droite"
    info_label.value = "Cliquez pour tracer la droite y = θ"

def on_click(b):
    if etat["phase"] == "manuel":
        afficher_point(etat["frame"])
        etat["frame"] += 1
        if etat["frame"] < seuil_manuel:
            info_label.value = f"Cliquez pour afficher le point {etat['frame'] + 1}"
        else:
            etat["phase"] = "auto"
            info_label.value = "Lancement automatique..."
            phase_automatique()

    elif etat["phase"] == "droite":
        for ax in axes.flat:
            ax.axhline(y=theta, color='red', linestyle='--')
        fig.canvas.draw_idle()
        etat["phase"] = "fini"
        info_label.value = "Terminé."
        bouton.disabled = True

bouton.on_click(on_click)

plt.show()

**Exemple 2.3 (suite)**

Considérons l'estimateur $\widehat{\theta }_{n,1}=2\overline{X}_n$.

1. Calculer le biais de $\widehat{\theta }_{n,1}$.
2. Calculer le risque quadratique de $\widehat{\theta }_{n,1}$.

*Solution*

1. Calcul du biais
   
$$
\begin{aligned}
\mathbb{E}_\theta (\hat{\theta}_{n,1})
&= \frac{2}{n} \sum_{i=1}^n \mathbb{E}_\theta (X_i) \text{ par linéarité de l'intégrale} \\
&= \frac{2}{n} \sum_{i=1}^n \frac{\theta}{2} \text{ car } X_i \sim \mathcal{U}([0; \theta]) \\
&= \frac{2}{n} \cdot \frac{n \theta}{2} = \theta
\end{aligned}
$$

Donc $b_\theta (\hat{\theta}_{n,1}) = \mathbb{E}_\theta (\hat{\theta}_{n,1}) - \theta = 0$.

$\hat{\theta}_{n,1}$ a un biais nul, il est donc sans biais de $\theta$.

---

2. Calcul du risque
   
$$
\begin{aligned}
r_\theta (\hat{\theta}_{n,1})
&= \mathbb{E}_\theta \left( \hat{\theta}_{n,1} - \theta \right)^2 \\
&= \mathbb{E}_\theta \left( \hat{\theta}_{n,1} - \mathbb{E}_\theta (\hat{\theta}_{n,1}) \right)^2 \text{ car } \hat{\theta}_{n,1} \text{ est sans biais de } \theta \\
&= \mathbb{V}_\theta (\hat{\theta}_{n,1}) \\
&= \frac{4}{n^2} \sum_{i=1}^n \mathbb{V}_\theta (X_i) \text{ d'après les propriétés de la variance} \\
&= \frac{4}{n^2} \sum_{i=1}^n \frac{\theta^2}{12} \text{ car } X_i \sim \mathcal{U}([0; \theta]) \\
&= \frac{\theta^2}{3n}
\end{aligned}
$$

Puisque $\lim_{n \to +\infty} \frac{\theta^2}{3n} = 0$, $\hat{\theta}_{n,1}$ converge vers $\theta$.

Cela signifie que l'erreur d'estimation est d'autant plus faible que la taille de l'échantillon est grande.

**Exemple 2.3 (suite)**

Calculer l'estimateur par la méthode des moments du temps d'attente.

*Solution*

Le temps d'attente (en s) étant une variable al\'{e}atoire suivant une loi $\mathcal{U}\left(\left[ 0;\theta \right]\right) $, chaque $X_i$ de l'échantillon suit une loi $\mathcal{U}\left(\left[ 0;\theta \right]\right) $.

Ainsi, pour tout $\theta \in \Theta $, $\mathbb{E}_{\theta }\left(X_i\right)  =\frac{\theta}{2}$.
En notant $\varphi\left(\theta\right)=\frac{\theta}{2}$, on a $\varphi^{-1}\left(\theta\right)=2\theta$ donc l'estimateur par la méthode des moments est $2\overline{X}_n$, qui n'est autre que $\widehat{\theta}_{n,1}$.

**Exemple 2.4 (Loi de Bernoulli)**

Soit $\left(X_1,\cdots,X_n\right)$ un $n$-échantillon de la loi de Bernoulli $\mathcal{B}\left(p\right)$.

Calculer l’estimateur par la méthode des moments de $p$.

*Solution*

Pour tout $p \in \left[0;1\right] $, $\mathbb{E}_{p}\left(X_i\right)=p$ donc l'estimateur par la méthode des moments de $p$ est $\overline{X}_n$ qui n'est autre que la fréquence $F_n$ définie dans la définition 2.4. 

**Exemple 2.5 (Loi normale)**

Soit $\left(X_1,\cdots,X_n\right)$ un $n$-échantillon de la loi normale $\mathcal{N}\left(\mu,\sigma^2\right)$.

Calculer l'estimateur par la méthode des moments de  $\left(\mu,\sigma^2\right)$.

*Solution*

Pour tout $\left(\mu,\sigma^2\right) \in \mathbb{R}\times \mathbb{R}_+ $, $\mathbb{E}_{\left(\mu,\sigma^2\right)}\left(X_i\right)=\mu$ et $\mathbb{V}_{\left(\mu,\sigma^2\right)}\left(X_i\right)=\sigma^2$ donc l'estimateur par la méthode des moments de $\left(\mu,\sigma^2\right)$ est $\left(\overline{X}_n,S^2_n\right)$. 

**Exemple 2.4 (Loi de Bernoulli suite)**

Soit $\left(X_1,\cdots,X_n\right)$ un $n$-échantillon de la loi de Bernoulli $\mathcal{B}\left(p\right)$.

Calculer l'estimateur par la méthode du maximum de vraisemblance de $p$.

*Solution*

Pour tout $p \in \left[0;1\right] $, la fonction de vraisemblance est

$$
\begin{aligned}
\mathcal{L}_{p}\left( x_{1},\cdots ,x_{n}\right) = 
\prod\limits_{i=1}^{n}\mathbb{P}_{p }\left( X_{i}=x_{i}\right)=\prod\limits_{i=1}^{n}p^{x_{i}}(1-p)^{1-x_i}=p^{\sum\limits_{i=1}^n x_{i}}(1-p)^{\sum\limits_{i=1}^n\left(1-x_i\right)}
\end{aligned}
$$

donc la fonction de log-vraisemblance est 

$$
\begin{aligned}
\ln\mathcal{L}_{p}\left( x_{1},\cdots ,x_{n}\right) =\sum\limits_{i=1}^n x_{i}\ln\left(p\right)+\left(n-\sum\limits_{i=1}^n x_i\right)\ln\left(1-p\right)
\end{aligned}
$$
 
Ainsi,

$$
\begin{aligned}
&&\frac{\partial}{\partial p}\ln\mathcal{L}_{p}\left( x_{1},\cdots ,x_{n}\right) =\frac{\sum\limits_{i=1}^n x_{i}}{p}-\frac{n-\sum\limits_{i=1}^n x_i}{1-p}=\frac{\sum\limits_{i=1}^n x_{i}-np}{p\left(1-p\right)}=0 \\
\Leftrightarrow  &&p=\overline{x}_n
\end{aligned}
$$

donc l'estimateur par maximum de vraisemblance de $p$ est $\overline{X}_n$ qui n'est autre que la fréquence $F_n$ définie dans la définition 2.4. 

**Exemple 2.5 (Loi normale suite)**

Soit $\left(X_1,\cdots,X_n\right)$ un $n$-échantillon de la loi normale $\mathcal{N}\left(\mu,\sigma^2\right)$.

Calculer l'estimateur par la méthode du maximum de vraisemblance de $\left(\mu,\sigma^2\right)$.

*Solution*

Pour tout $\left(\mu,\sigma^2\right) \in \mathbb{R}\times\mathbb{R}_+^*$, la fonction de vraisemblance est

$$
\begin{aligned}
\mathcal{L}_{\left(\mu,\sigma^2\right)}\left( x_{1},\cdots ,x_{n}\right) = 
\prod\limits_{i=1}^{n}f_{\left(\mu,\sigma^2\right) }\left( x_{i}\right)=\prod\limits_{i=1}^{n}\frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{\left(x_i-\mu\right)^2}{2\sigma^2}}=\frac{1}{\left(\sigma\sqrt{2\pi}\right)^n}e^{-\frac{1}{2\sigma^2}\sum\limits_{i=1}^n \left(x_i-\mu\right)^2}
\end{aligned}
$$

donc la fonction de log-vraisemblance est 

$$
\begin{aligned}
\ln\mathcal{L}_{\left(\mu,\sigma^2\right)}\left( x_{1},\cdots ,x_{n}\right) =-\frac{n}{2}\ln\left(\sigma^2\right)-\frac{n}{2}\ln\left(2\pi\right)-\frac{1}{2\sigma^2}\sum\limits_{i=1}^n \left(x_i-\mu\right)^2
\end{aligned}
$$

Les équations de vraisemblance sont alors

$$
\begin{aligned}
&&\left\{ 
\begin{array}{c}
\frac{\partial}{\partial \mu}\ln\mathcal{L}_{\left(\mu,\sigma^2\right)}\left( x_{1},\cdots ,x_{n}\right) =-\frac{1}{2\sigma^2}\sum\limits_{i=1}^n -2\left(x_i-\mu\right)=\frac{1}{\sigma^2}\left(\sum\limits_{i=1}^n x_i-n\mu\right)=0 \\ 
\frac{\partial}{\partial \sigma^2}\ln\mathcal{L}_{\left(\mu,\sigma^2\right)}\left( x_{1},\cdots ,x_{n}\right) =-\frac{n}{2\sigma^2}+\frac{1}{2\sigma^4}\sum\limits_{i=1}^n \left(x_i-\mu\right)^2=0 
\end{array}%
\right.  \\
\Leftrightarrow  &&\left\{ 
\begin{array}{c}
\mu=\overline{x}_n \\ 
\sigma^2=s^2_n%
\end{array}%
\right. 
\end{aligned}
$$

donc l'estimateur par maximum de vraisemblance de $\left(\mu,\sigma^2\right)$ est $\left(\overline{X}_n,S_n^2\right)$. 

**Exemple 2.4 (Loi de Bernoulli suite)**

Soit $\left(X_1,\cdots,X_n\right)$ un $n$-échantillon de la loi de Bernoulli $\mathcal{B}\left(p\right)$.

Déterminer un estimateur efficace de $p$.

*Solution*

Pour tout $p \in \left[0;1\right] $, $\mathbb{E}_p\left(X_i\right)=p$ et $\mathbb{V}_p\left(X_i\right)=p\left(1-p\right)$.

On a montré précédemment que la fréquence $F_n=\overline{X}_n$ est l'estimateur par la méthode des moments et par maximum de vraisemblance de $p$.

D'après la propriété 2.2, $F_n=\overline{X}_n$ est sans biais et convergent vers $p$.

Rappelons que $\mathbb{V}_p\left(F_n\right)=\frac{\mathbb{V}_p\left(X_i\right)}{n}=\frac{p\left(1-p\right)}{n}$. 

On a vu précédemment que

$$
\begin{aligned}
\frac{\partial }{\partial p}\ln \mathcal{L}_{p}\left( X_{1},\cdots ,X_{n}\right) =
\frac{\sum\limits_{i=1}^{n}x_{i}-np}{p\left( 1-p\right) }
\end{aligned}
$$

donc

$$
\begin{aligned}
\frac{\partial ^{2}}{\partial p^{2}}\ln \mathcal{L}_{p}\left( X_{1},\cdots,X_{n}\right) =
\frac{\sum\limits_{i=1}^{n}x_{i}-np}{p\left( 1-p\right) }=
\frac{-np\left( 1-p\right) -\left( \sum\limits_{i=1}^{n}x_{i}-np\right)}{p^{2}(1-p)^{2}}
\end{aligned}
$$

Ainsi, l’information de Fisher de $\bar{X}_n$ vaut :

$$
\begin{aligned}
I_n &= \mathbb{E}_p \left( \frac{\partial^2}{\partial p^2} \ln \mathcal{L}_p(X_1, \dots, X_n) \right) \\
&= \frac{n}{p(1-p)} + \frac{(1-2p)}{p^2(1-p)^2} \mathbb{E}_p \left( \sum_{i=1}^n x_i - np \right) \\
&= \frac{n}{p(1-p)}
\end{aligned}
$$

et alors BRC$(p) = \frac{p(1-p)}{n}$.

Puisque BRC$(p) = \mathbb{V}_p(\bar{X}_n) $, la fréquence $ \bar{F}_n $ de $p $ est efficace et donc sans biais optimal.

**Exemple 2.5 (Loi normale suite)**

Soit $\left(X_1,\cdots,X_n\right)$ un $n$-échantillon de la loi normale $\mathcal{N}\left(\mu,\sigma^2\right)$.

Déterminer un estimateur efficace de $\left(\mu,\sigma^2\right)$.

*Solution*

On a montré précédemment que $\left(\overline{X}_n,S_n^2\right)$ est l'estimateur par la méthode des moments et par maximum de vraisemblance de $\left(\mu,\sigma^2\right)$. 


On peut montrer sans difficulté que $\overline{X}_n$ est un estimateur sans biais optimal de $\mu$. Par contre, on a montré que $S_n^2$ est un estimateur biaisé de $\sigma^2$ donc il n'est pas sans biais optimal. On peut néanmoins montrer que $S_{n,c}^2$ est un estimateur sans biais optimal de $\sigma^2$.

Concernant l'écart-type $\sigma$, on peut montrer que $\sqrt{\frac{n-1}{2}}\frac{\Gamma\left(\frac{n-1}{2}\right)}{\Gamma\left(\frac{n}{2}\right)}S_{n,c}$ est un estimateur sans biais optimal de $\sigma$.

**Exemple 2.1 (suite)**

Un constructeur automobile fabrique des moteurs diesel pour un certain modèle de voiture. La population est donc constituée de tous les moteurs produits. L'ingénieur qualité s'intéresse à la pollution en NOx des moteurs. 

Il prélève un échantillon de taille 7 dont la réalisation fournit les résultats suivants : (78, 82, 81, 82 ,79, 80, 78).

Calculer un intervalle de confiance au niveau de confiance 95\% du taux de pollution moyen en NOx des moteurs.

**Exemple 2.2 (suite)**

Une entreprise automobile s'intéresse à une pièce fondamentale constitutive des moteurs produits. 

Elle prélève un échantillon de taille 7 dont la réalisation fournit les résultats suivants : (1,0,0,1,0,0,0) où 1 signifie que la pièce est défectueuse et 0 sinon. 

Calculer un intervalle de confiance au niveau de confiance 95\% de la proportion de pièces défectueuses.